# RAGAS Evaluation

- **Faithfulness**
- **Answer Relevance**

In [1]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")


True

### 1. Generate Pipeline Outputs

In [2]:
from main import agentic_rag_app
from datasets import Dataset

def get_pipeline_data(query: str):
    initial_state = {
        "query": query,
        "original_query": query,
        "namespace": "default_namespace",
        "documents": [],
        "final_context_strips": [],
        "needs_web_search": False,
        "answer": "",
        "retries": 0,
        "hallucination_retries": 0,
        "route": ""
    }
    result = agentic_rag_app.invoke(initial_state)
    
    # lists for context strings
    context = result.get("final_context_strips", [])
    if not context:
        context = ["LLM General Knowledge/No specific context found."]
        
    return result.get("answer", ""), context

queries = [
    "What are two specific complementary graphs constructured during the dual graph construction phase?",
    "what are the two multimodal document question answering (DQA) benchmarks used to test the model performance?",
    "What is the current Delhi Temperature?",
    "Has the corresponding author of RAG-Anything, Chao Huang from the University of Hong Kong, announced any new multimodal framework updates or papers in the past week?",
    "Can you explain the general concept of 'Retrieval-Augmented Generation' and why it is necessary for LLM?",
    "write a Python function using numpy to calculate the cosine similarity between two 1D array embeddings, which is a common metric used in semantic matching"
]

data = {"question": [], "answer": [], "contexts": []}

for q in queries:
    print(f"Evaluating query: {q}")
    ans, ctx = get_pipeline_data(q)
    data["question"].append(q)
    data["answer"].append(ans)
    data["contexts"].append(ctx)

dataset = Dataset.from_dict(data)
dataset.to_pandas()

c:\Users\amanm\Desktop\learning\developer-chat-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Index 'agenticrag' already exists


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1323.31it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluating query: What are two specific complementary graphs constructured during the dual graph construction phase?
2026-03-29 17:11:20 - src.adaptive_router - INFO - Query 'What are two specific complementary graphs constructured during the dual graph construction phase?' classified to route: 'vectorstore'
2026-03-29 17:11:20 - src.workflows.crag - INFO - Using Serverless Pinecone Reranking Model
2026-03-29 17:11:22 - src.retrieval.reranker - INFO - Reranker returned 5 hits (top_n=5)
2026-03-29 17:12:03 - src.workflows.crag - INFO - CRAG evaluate: 13 relevant strips found. needs_web_search=False
2026-03-29 17:12:07 - src.workflows.self_rag - INFO - Hallucination Score: yes
2026-03-29 17:12:10 - src.workflows.self_rag - INFO - Answer Quality Score: yes
Evaluating query: what are the two multimodal document question answering (DQA) benchmarks used to test the model performance?
2026-03-29 17:12:12 - src.adaptive_router - INFO - Query 'what are the two multimodal document question answe

,question,answer,contexts
0,What are two specific complementary graphs con...,The context does not explicitly mention the na...,"[consistent representation., 2.2.1 DUAL-GRAPH ..."
1,what are the two multimodal document question ...,The two multimodal Document Question Answering...,"[current multimodal RAG systems., It delivers ..."
2,What is the current Delhi Temperature?,The current Delhi temperature is 31°C.,[[Web Search Results]: 17 hours ago -Current N...
3,"Has the corresponding author of RAG-Anything, ...","Based on the provided context, there is no inf...","[[Web Search Results]: October 14, 2025 -From ..."
4,Can you explain the general concept of 'Retrie...,Retrieval-Augmented Generation (RAG) is a conc...,[LLM General Knowledge/No specific context fou...
5,write a Python function using numpy to calcula...,### Cosine Similarity Calculation using NumPy\...,[Our cross-modal hybrid retrieval mechanism st...


### 2. Configure RAGAS Custom Models

In [3]:
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.embeddings import HuggingFaceEmbeddings
from src.generation.generator import llm

ragas_llm = LangchainLLMWrapper(llm, is_finished_parser=lambda x: True)
ragas_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))
faithfulness.llm = ragas_llm
answer_relevancy.llm = ragas_llm
answer_relevancy.embeddings = ragas_emb
answer_relevancy.strictness = 1

C:\Users\amanm\AppData\Local\Temp\ipykernel_17448\1168877654.py:1: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
C:\Users\amanm\AppData\Local\Temp\ipykernel_17448\1168877654.py:1: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy
C:\Users\amanm\AppData\Local\Temp\ipykernel_17448\1168877654.py:7: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'

### 3. Run RAGAS Evaluation

In [4]:
from ragas import evaluate

results = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy],
    llm=ragas_llm,
    embeddings=ragas_emb,
)

df_results = results.to_pandas()
print("Evaluation complete!")
df_results

Evaluating: 100%|██████████| 12/12 [00:45<00:00,  3.80s/it]


Evaluation complete!


,user_input,retrieved_contexts,response,faithfulness,answer_relevancy
0,What are two specific complementary graphs con...,"[consistent representation., 2.2.1 DUAL-GRAPH ...",The context does not explicitly mention the na...,1.000000,0.979460
1,what are the two multimodal document question ...,"[current multimodal RAG systems., It delivers ...",The two multimodal Document Question Answering...,1.000000,1.000000
2,What is the current Delhi Temperature?,[[Web Search Results]: 17 hours ago -Current N...,The current Delhi temperature is 31°C.,1.000000,0.984616
3,"Has the corresponding author of RAG-Anything, ...","[[Web Search Results]: October 14, 2025 -From ...","Based on the provided context, there is no inf...",0.333333,0.811840
4,Can you explain the general concept of 'Retrie...,[LLM General Knowledge/No specific context fou...,Retrieval-Augmented Generation (RAG) is a conc...,0.000000,0.750995
5,write a Python function using numpy to calcula...,[Our cross-modal hybrid retrieval mechanism st...,### Cosine Similarity Calculation using NumPy\...,0.250000,0.887317


In [5]:
df_results.to_csv("eval_results.csv")